# Fine-Tuning GPT-2 for Creative Story Generation

This notebook demonstrates how to fine-tune GPT-2 for creative story generation **without using any external dataset file**.

A small in-memory dataset is created only for demonstration and explanation purposes.

## 1. Install Required Libraries
These libraries are required to load the model and perform fine-tuning.

In [ ]:
!pip install transformers datasets torch accelerate

## 2. Import Required Libraries
We import Hugging Face Transformers, Datasets, and PyTorch.

In [ ]:

from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import Dataset
import torch


## 3. Load Pre-trained GPT-2 Model and Tokenizer
GPT-2 is used as the base model for story generation.

In [ ]:

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# GPT-2 does not define a padding token by default
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))


## 4. Create an In-Memory Sample Dataset
No external dataset file is required.

In [ ]:

stories = [
    "Once upon a time, a lonely robot learned how to feel emotions.",
    "The moon whispered secrets to the ocean under the silver night sky.",
    "In a forgotten village, a child discovered a book that could predict the future.",
    "The last dragon flew across the burning horizon, guarding ancient magic."
]

dataset = Dataset.from_dict({"text": stories})
dataset


## 5. Tokenization
Convert story text into tokens that GPT-2 understands.

In [ ]:

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])


## 6. Training Configuration
Define hyperparameters for fine-tuning.

In [ ]:

training_args = TrainingArguments(
    output_dir="./gpt2-story-generator",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    logging_steps=10,
    save_steps=500,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available()
)


## 7. Fine-Tuning the Model
The Trainer API manages training.

In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()


## 8. Save the Fine-Tuned Model
Save the trained model and tokenizer.

In [ ]:

model.save_pretrained("./gpt2-story-generator")
tokenizer.save_pretrained("./gpt2-story-generator")


## 9. Story Generation
Generate a creative story using the fine-tuned model.

In [ ]:

prompt = "The abandoned spaceship drifted silently through space"
inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_length=200,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


## 10. Conclusion
This notebook demonstrates GPT-2 fine-tuning for creative story generation without relying on any external dataset file.

In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the fine-tuned model and tokenizer
MODEL_PATH = "./gpt2-story-generator"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH)

# Set model to evaluation mode
model.eval()

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Test prompt
prompt = "The scientist opened the ancient book and suddenly"

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate story
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_length=250,
        temperature=1.0,     # controls creativity
        top_k=50,            # limits random choices
        top_p=0.95,          # nucleus sampling
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and print output
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("Generated Story:\n")
print(generated_text)


c:\Users\januu\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generated Story:

The scientist opened the ancient book and suddenly saw it and said:

[I]n it the Book the Lord brought forth a new chapter, which shall be fulfilled.
